# One-time GitHub bootstrap — Cinematic Short Engine

Use this notebook only to publish the implementation ZIP into your own repository when the connected GitHub tool cannot access the repo. It does **not** ask you to paste a token in code.

**Before running:** in Google Colab → Secrets (key icon), create `GITHUB_TOKEN` from a fine-grained Personal Access Token that grants **Contents: Read and write** for `havd58863-ui/cinematic-short-engine`. Never put the token in a notebook cell or repository. Then run all cells and upload the implementation ZIP when prompted.

The uploader creates one Git commit containing all source files (or preserves existing paths when the repo already has a branch). It never deletes unrelated files and skips generated media, model weights, caches, and secrets.


In [ ]:
from google.colab import files, userdata
from pathlib import Path
import zipfile, shutil, os, sys

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise RuntimeError('Upload the cinematic-short-engine-implementation.zip file.')
zip_path = Path('/content') / zip_names[0]
extract_root = Path('/content/cinematic-short-engine-bootstrap')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as zf:
    # Guard against path traversal when extracting an archive.
    for member in zf.infolist():
        destination = (extract_root / member.filename).resolve()
        if extract_root.resolve() not in destination.parents and destination != extract_root.resolve():
            raise RuntimeError('Unsafe path in uploaded ZIP; extraction stopped.')
    zf.extractall(extract_root)
roots = [p.parent for p in extract_root.rglob('pyproject.toml')]
if not roots:
    raise FileNotFoundError('The ZIP does not contain the project pyproject.toml.')
PROJECT_ROOT = roots[0]
print('Project files loaded:', PROJECT_ROOT)


In [ ]:
# Read the token from Colab Secrets. Its value is never printed.
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Add a Colab Secret named GITHUB_TOKEN with repository Contents read/write permission, then rerun this cell.') from exc
if not token:
    raise RuntimeError('The GITHUB_TOKEN Colab Secret is empty or unavailable.')
os.environ['GITHUB_TOKEN'] = token
os.environ['GITHUB_REPOSITORY'] = 'havd58863-ui/cinematic-short-engine'
subprocess_code = [sys.executable, str(PROJECT_ROOT / 'tools' / 'push_to_github.py'), '--root', str(PROJECT_ROOT), '--repo', os.environ['GITHUB_REPOSITORY']]
import subprocess
result = subprocess.run(subprocess_code, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout[-12000:])
if result.returncode:
    # Server errors can mention permission, missing repository, or an absent default branch. Token itself is never shown.
    print((result.stderr or '')[-4000:])
    raise RuntimeError('GitHub bootstrap did not finish. Review the API error above; do not paste the token into chat.')


## After upload

Open the main production notebook from your repository:

`https://colab.research.google.com/github/havd58863-ui/cinematic-short-engine/blob/main/notebook/run_pipeline.ipynb`

If your repository's default branch is not `main`, replace `main` with the branch shown in the successful push result. The production notebook is reusable; generated jobs and model weights are stored in Google Drive, not GitHub.
